In [ ]:
import io
import os
from os import path
import tarfile
import urllib.request

import ase.io
import numpy as np

In [ ]:
URL = "https://springernature.figshare.com/ndownloader/files/3195389"
FILENAME = "dsgdb9nsd.xyz.tar.bz2"
DATA_DIR = "/tmp/qm9/"

In [ ]:
if not path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

In [ ]:
out_file = path.join(DATA_DIR, FILENAME)
url = path.join(URL, FILENAME)
if not path.isfile(out_file):
    urllib.request.urlretrieve(url, out_file)
    print(f"downloaded {url} to {DATA_DIR}")

In [ ]:
def read_qm9(file_handle):
    # Format description can be found here:
    # https://springernature.figshare.com/articles/dataset/Readme_file_Data_description_for_Quantum_chemistry_structures_and_properties_of_134_kilo_molecules_/1057641?backTo=%2Fcollections%2FQuantum_chemistry_structures_and_properties_of_134_kilo_molecules%2F978904&file=3195392
    labels = [
        "tag",
        "index",
        "A",
        "B",
        "C",
        "mu",
        "alpha",
        "homo",
        "lumo",
        "gap",
        "r2",
        "zpve",
        "U0",
        "U",
        "H",
        "G",
        "Cv",
    ]

    file_handle = io.TextIOWrapper(file_handle, encoding="utf-8")
    lines = file_handle.readlines()
    num_atoms = int(lines[0].strip())
    properties = lines[1].split()  # Contains properties like energy, dipole moment, etc.
    atoms = [line.split() for line in lines[2 : 2 + num_atoms]]

    # Parse atomic symbols and positions
    species = [atom[0] for atom in atoms]
    coords = [[float(x.replace("*^", "E")) for x in atom[1:4]] for atom in atoms]

    molecule = ase.Atoms(positions=coords, symbols=species)
    # Now add the properties
    for label, property in zip(labels, properties):
        molecule.arrays[label] = property

    return molecule

In [ ]:
structures = []
with tarfile.open(out_file, "r", encoding="utf-8") as tar:
    for member in tar:
        with tar.extractfile(member) as fh:
            print(member.name)
            structures.append(read_qm9(fh))

In [ ]:
structures[0].arrays

In [ ]:
atomline = "2.1997*^-6	 1.4462618059	 0.0098312216	-0.335446"

In [ ]:
np.fromstring(atomline)

In [ ]:
atomline

In [ ]:
atomlinedd